# AI vs. Real Image Classifier

This Google Colab notebook trains an EfficientNetV2-S model to classify images as AI-generated (`FAKE`) or real (`REAL`).

> **Colab required:** the notebook uses `google.colab`, mounted Google Drive paths, and Colab's upload interface. GitHub hosts the source, but it does not execute the notebook. See `README.md` for complete setup instructions.

Before running the cells:

1. Upload the dataset as `My Drive/ai_vs_real_images/Compressed.zip`.
2. Select a GPU runtime in Colab.
3. Run every cell in order.

Checkpoints and metrics are saved under `My Drive/ai_vs_real_model_checkpoints`.


## 1. Mount Drive & Configure the Project

In [ ]:
import json
import os
import shutil
import tempfile
import zipfile
from datetime import datetime

from google.colab import drive
from tqdm.notebook import tqdm

drive.mount('/content/drive')

DRIVE_IMAGES_FOLDER = '/content/drive/MyDrive/ai_vs_real_images'
DRIVE_BASE = '/content/drive/MyDrive/ai_vs_real_model_checkpoints'
DRIVE_ZIP = os.path.join(DRIVE_IMAGES_FOLDER, 'Compressed.zip')

LOCAL_DATASET_ROOT = '/content/dataset_subset'
LOCAL_TRAIN_ROOT = os.path.join(LOCAL_DATASET_ROOT, 'TRAIN')
LOCAL_VAL_ROOT = os.path.join(LOCAL_DATASET_ROOT, 'VAL')
LOCAL_TEST_ROOT = os.path.join(LOCAL_DATASET_ROOT, 'TEST')

LOCAL_TRAIN_REAL = os.path.join(LOCAL_TRAIN_ROOT, 'REAL')
LOCAL_TRAIN_FAKE = os.path.join(LOCAL_TRAIN_ROOT, 'FAKE')
LOCAL_TEST_REAL = os.path.join(LOCAL_TEST_ROOT, 'REAL')
LOCAL_TEST_FAKE = os.path.join(LOCAL_TEST_ROOT, 'FAKE')

RUNS_BASE = os.path.join(DRIVE_BASE, 'runs')
LATEST_RUN_PATH = os.path.join(DRIVE_BASE, 'latest_run.json')
DATASET_CACHE_PATH = os.path.join(DRIVE_BASE, 'dataset_cache.json')

RESUME_FROM_LAST = True
TARGET_TOTAL_EPOCHS = 12
VAL_FRACTION = 0.20
RANDOM_SEED = 42

os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(RUNS_BASE, exist_ok=True)

def atomic_write_text(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fd, temp_path = tempfile.mkstemp(
        dir=os.path.dirname(path),
        prefix='.tmp_',
        suffix=os.path.splitext(path)[1] or '.tmp',
    )
    try:
        with os.fdopen(fd, 'w', encoding='utf-8') as handle:
            handle.write(text)
            handle.flush()
            try:
                os.fsync(handle.fileno())
            except OSError:
                pass
        os.replace(temp_path, path)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

def atomic_write_json(path, payload):
    atomic_write_text(path, json.dumps(payload, indent=2))

def load_json(path, default=None):
    if not os.path.exists(path):
        return default
    try:
        with open(path, 'r', encoding='utf-8') as handle:
            return json.load(handle)
    except (OSError, json.JSONDecodeError):
        return default

def fingerprint_file(path):
    stat = os.stat(path)
    return {
        'path': path,
        'size': stat.st_size,
        'mtime': int(stat.st_mtime),
    }

latest_run = load_json(LATEST_RUN_PATH, default={}) if RESUME_FROM_LAST else {}
ACTIVE_RUN_DIR = latest_run.get('active_run_dir')
if ACTIVE_RUN_DIR and not os.path.isdir(ACTIVE_RUN_DIR):
    ACTIVE_RUN_DIR = None
if not ACTIVE_RUN_DIR:
    ACTIVE_RUN_DIR = os.path.join(RUNS_BASE, datetime.now().strftime('%Y%m%d_%H%M%S'))
os.makedirs(ACTIVE_RUN_DIR, exist_ok=True)

LAST_CKPT_PATH = os.path.join(ACTIVE_RUN_DIR, 'last_checkpoint.pth')
BEST_CKPT_PATH = os.path.join(ACTIVE_RUN_DIR, 'best_checkpoint.pth')
METRICS_JSONL_PATH = os.path.join(ACTIVE_RUN_DIR, 'epoch_metrics.jsonl')
METRICS_CSV_PATH = os.path.join(ACTIVE_RUN_DIR, 'epoch_metrics.csv')
RUN_CONFIG_PATH = os.path.join(ACTIVE_RUN_DIR, 'run_config.json')
RUN_SUMMARY_PATH = os.path.join(ACTIVE_RUN_DIR, 'run_summary.json')
LEGACY_BEST_MODEL_PATH = os.path.join(DRIVE_BASE, 'best_model.pth')

atomic_write_json(
    LATEST_RUN_PATH,
    {
        'active_run_dir': ACTIVE_RUN_DIR,
        'updated_at': datetime.now().isoformat(),
        'target_total_epochs': TARGET_TOTAL_EPOCHS,
    },
)

if os.path.exists(DRIVE_ZIP):
    current_fingerprint = fingerprint_file(DRIVE_ZIP)
    cached_state = load_json(DATASET_CACHE_PATH, default={}) or {}
    cache_hit = (
        cached_state.get('zip_fingerprint') == current_fingerprint
        and os.path.isdir(LOCAL_TRAIN_ROOT)
    )

    if cache_hit:
        print('Dataset cache is current. Skipping extraction.')
    else:
        if os.path.exists(LOCAL_DATASET_ROOT):
            shutil.rmtree(LOCAL_DATASET_ROOT)
        os.makedirs(LOCAL_DATASET_ROOT, exist_ok=True)

        print(f'Extracting dataset from: {DRIVE_ZIP}')
        with zipfile.ZipFile(DRIVE_ZIP, 'r') as zip_ref:
            members = zip_ref.infolist()
            for member in tqdm(members, desc='Extracting files'):
                zip_ref.extract(member, LOCAL_DATASET_ROOT)

        extracted_items = [item for item in os.listdir(LOCAL_DATASET_ROOT) if not item.startswith('.')]
        if len(extracted_items) == 1:
            nested_path = os.path.join(LOCAL_DATASET_ROOT, extracted_items[0])
            if os.path.isdir(nested_path):
                print(f'Flattening nested folder: {extracted_items[0]}')
                for item in os.listdir(nested_path):
                    shutil.move(os.path.join(nested_path, item), LOCAL_DATASET_ROOT)
                os.rmdir(nested_path)

        atomic_write_json(
            DATASET_CACHE_PATH,
            {
                'zip_fingerprint': current_fingerprint,
                'dataset_root': LOCAL_DATASET_ROOT,
                'updated_at': datetime.now().isoformat(),
            },
        )
        print('Dataset extraction complete.')
else:
    raise FileNotFoundError(f'Could not find dataset archive at {DRIVE_ZIP}')

atomic_write_json(
    RUN_CONFIG_PATH,
    {
        'resume_from_last': RESUME_FROM_LAST,
        'target_total_epochs': TARGET_TOTAL_EPOCHS,
        'val_fraction': VAL_FRACTION,
        'random_seed': RANDOM_SEED,
        'drive_zip': DRIVE_ZIP,
        'dataset_root': LOCAL_DATASET_ROOT,
        'run_dir': ACTIVE_RUN_DIR,
        'updated_at': datetime.now().isoformat(),
    },
)

print('Setup complete.')
print(f'Active run directory: {ACTIVE_RUN_DIR}')

## 2. Extract and Verify the Dataset

In [ ]:
import os

def count_files(path):
    if not os.path.exists(path): return 0
    return len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])

print('Checking local dataset structure...')
print(f'TRAIN REAL: {count_files(LOCAL_TRAIN_REAL)} images')
print(f'TRAIN FAKE: {count_files(LOCAL_TRAIN_FAKE)} images')
print(f'TEST REAL: {count_files(LOCAL_TEST_REAL)} images')
print(f'TEST FAKE: {count_files(LOCAL_TEST_FAKE)} images')

if count_files(LOCAL_TRAIN_REAL) > 0:
    print('\n Local dataset is ready for training.')
else:
    print('\n Local dataset not found. Please re-run Cell 01 to unzip the files.')

## 3. DataLoaders

In [ ]:
import io
import os

import numpy as np
import torch
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

ENABLE_JPEG_AUG = True
TRAIN_IMAGE_SIZE = 224
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 256

class RandomJPEGCompression:
    def __init__(self, quality=(65, 95)):
        self.quality = quality

    def __call__(self, img):
        quality = torch.randint(self.quality[0], self.quality[1], (1,)).item()
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert('RGB')

train_transforms = [
    transforms.RandomResizedCrop(TRAIN_IMAGE_SIZE, scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.03)], p=0.3),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.1),
]
if ENABLE_JPEG_AUG:
    train_transforms.append(transforms.RandomApply([RandomJPEGCompression()], p=0.05))
train_transforms.extend([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
train_transform = transforms.Compose(train_transforms)

eval_transform = transforms.Compose([
    transforms.Resize(TRAIN_IMAGE_SIZE + 12, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(TRAIN_IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

if not os.path.isdir(LOCAL_TRAIN_ROOT):
    raise RuntimeError(f'TRAIN directory was not found at {LOCAL_TRAIN_ROOT}')

train_source_dataset = datasets.ImageFolder(root=LOCAL_TRAIN_ROOT, transform=train_transform)
eval_source_dataset = datasets.ImageFolder(root=LOCAL_TRAIN_ROOT, transform=eval_transform)
class_names = train_source_dataset.classes
targets = np.array(train_source_dataset.targets)

def print_distribution(name, values):
    counts = np.bincount(values, minlength=len(class_names))
    summary = ', '.join(f'{class_names[idx]}={int(count)}' for idx, count in enumerate(counts))
    print(f'{name}: {summary}')

def build_loader(dataset, batch_size, shuffle=False, sampler=None):
    loader_kwargs = {
        'batch_size': batch_size,
        'shuffle': shuffle if sampler is None else False,
        'sampler': sampler,
        'num_workers': NUM_WORKERS,
        'pin_memory': torch.cuda.is_available(),
        'drop_last': False,
    }
    if NUM_WORKERS > 0:
        loader_kwargs['persistent_workers'] = True
        loader_kwargs['prefetch_factor'] = 2
    return DataLoader(dataset, **loader_kwargs)

if os.path.isdir(LOCAL_VAL_ROOT) and any(os.scandir(LOCAL_VAL_ROOT)):
    train_dataset = train_source_dataset
    train_eval_dataset = datasets.ImageFolder(root=LOCAL_TRAIN_ROOT, transform=eval_transform)
    val_dataset = datasets.ImageFolder(root=LOCAL_VAL_ROOT, transform=eval_transform)
    train_targets = targets
    val_targets = np.array(val_dataset.targets)
    print('Using dedicated VAL split from dataset.')
else:
    train_indices, val_indices = train_test_split(
        np.arange(len(targets)),
        test_size=VAL_FRACTION,
        stratify=targets,
        random_state=RANDOM_SEED,
        shuffle=True,
    )
    train_dataset = Subset(train_source_dataset, train_indices.tolist())
    train_eval_dataset = Subset(eval_source_dataset, train_indices.tolist())
    val_dataset = Subset(eval_source_dataset, val_indices.tolist())
    train_targets = targets[train_indices]
    val_targets = targets[val_indices]
    print(f'Created stratified validation split from TRAIN with val_fraction={VAL_FRACTION:.2f}.')

test_dataset = None
test_targets = None
if os.path.isdir(LOCAL_TEST_ROOT) and any(os.scandir(LOCAL_TEST_ROOT)):
    test_dataset = datasets.ImageFolder(root=LOCAL_TEST_ROOT, transform=eval_transform)
    test_targets = np.array(test_dataset.targets)

class_counts = np.bincount(train_targets, minlength=len(class_names))
imbalance_ratio = class_counts.max() / max(1, class_counts.min())
train_sampler = None
if imbalance_ratio > 1.35:
    weights = torch.as_tensor(1.0 / class_counts[train_targets], dtype=torch.double)
    train_sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)
    print(f'Enabled WeightedRandomSampler because class imbalance ratio is {imbalance_ratio:.2f}.')

num_cpus = os.cpu_count() or 2
NUM_WORKERS = min(4, max(2, num_cpus - 1)) if torch.cuda.is_available() else 0

train_loader = build_loader(train_dataset, TRAIN_BATCH_SIZE, shuffle=True, sampler=train_sampler)
train_eval_loader = build_loader(train_eval_dataset, EVAL_BATCH_SIZE)
val_loader = build_loader(val_dataset, EVAL_BATCH_SIZE)
test_loader = build_loader(test_dataset, EVAL_BATCH_SIZE) if test_dataset is not None else None

print_distribution('Train distribution', train_targets)
print_distribution('Val distribution', val_targets)
if test_targets is not None:
    print_distribution('Test distribution', test_targets)

print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')
if test_dataset is not None:
    print(f'Test samples: {len(test_dataset)}')
print(f'Dataloader workers: {NUM_WORKERS}')

## 4. Model, Optimizer & Load Checkpoint

In [ ]:
import random

import numpy as np
import torch
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, 'allow_tf32'):
        torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

BACKBONE_NAME = 'efficientnet_v2_s'
BACKBONE_LR = 5e-5
HEAD_LR = 3e-4
WEIGHT_DECAY = 1e-2
FREEZE_BACKBONE = False
LABEL_SMOOTHING = 0.1
MIXUP_ALPHA = 0.2
EARLY_STOP_PATIENCE = 5
VALIDATE_EVERY_N_EPOCHS = 1

def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_SEED)

def build_model(freeze_backbone=FREEZE_BACKBONE):
    if BACKBONE_NAME != 'efficientnet_v2_s':
        raise ValueError(f'Unsupported backbone: {BACKBONE_NAME}')
    model = models.efficientnet_v2_s(weights='DEFAULT')
    for parameter in model.parameters():
        parameter.requires_grad = not freeze_backbone
    in_features = model.classifier[1].in_features
    model.classifier = nn.Linear(in_features, 2)
    return model.to(device).to(memory_format=torch.channels_last)

def build_optimizer(model):
    backbone_params = [parameter for name, parameter in model.named_parameters() if 'classifier' not in name and parameter.requires_grad]
    head_params = [parameter for name, parameter in model.named_parameters() if 'classifier' in name and parameter.requires_grad]
    param_groups = []
    if backbone_params:
        param_groups.append({'params': backbone_params, 'lr': BACKBONE_LR})
    if head_params:
        param_groups.append({'params': head_params, 'lr': HEAD_LR})
    return torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

def build_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=2,
        min_lr=BACKBONE_LR * 0.2,
    )

def mixup(inputs, labels, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return inputs, labels, labels, 1.0
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    indices = torch.randperm(inputs.size(0), device=inputs.device)
    mixed_inputs = lam * inputs + (1 - lam) * inputs[indices]
    return mixed_inputs, labels, labels[indices], lam

def mixup_loss(loss_fn, predictions, labels_a, labels_b, lam):
    return lam * loss_fn(predictions, labels_a) + (1 - lam) * loss_fn(predictions, labels_b)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print(f'Backbone: {BACKBONE_NAME} | Target total epochs: {TARGET_TOTAL_EPOCHS} | Device: {device.type}')

## 5. Training Loop

In [ ]:
import csv
import io
import json
import os
import random
import time
from datetime import datetime

import numpy as np
import torch
from sklearn.metrics import f1_score
from torch.amp import GradScaler, autocast
from tqdm.notebook import tqdm

run_config = load_json(RUN_CONFIG_PATH, default={}) or {}
run_config.update({
    'backbone_name': BACKBONE_NAME,
    'backbone_lr': BACKBONE_LR,
    'head_lr': HEAD_LR,
    'weight_decay': WEIGHT_DECAY,
    'mixup_alpha': MIXUP_ALPHA,
    'label_smoothing': LABEL_SMOOTHING,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'validate_every_n_epochs': VALIDATE_EVERY_N_EPOCHS,
    'train_batch_size': TRAIN_BATCH_SIZE,
    'eval_batch_size': EVAL_BATCH_SIZE,
    'image_size': TRAIN_IMAGE_SIZE,
    'enable_jpeg_aug': ENABLE_JPEG_AUG,
    'track_train_metrics': True,
    'updated_at': datetime.now().isoformat(),
})
atomic_write_json(RUN_CONFIG_PATH, run_config)

def atomic_torch_save(payload, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temp_path = f'{path}.tmp'
    torch.save(payload, temp_path)
    os.replace(temp_path, path)

def append_jsonl(path, record):
    with open(path, 'a', encoding='utf-8') as handle:
        handle.write(json.dumps(record) + '\n')
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

def write_csv(path, records):
    if not records:
        return
    fieldnames = list(records[-1].keys())
    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=fieldnames)
    writer.writeheader()
    for record in records:
        writer.writerow({name: record.get(name) for name in fieldnames})
    atomic_write_text(path, buffer.getvalue())

def get_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['cuda'] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    if not state:
        return
    if 'python' in state:
        random.setstate(state['python'])
    if 'numpy' in state:
        np.random.set_state(state['numpy'])
    if 'torch' in state:
        torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and 'cuda' in state:
        torch.cuda.set_rng_state_all(state['cuda'])

def checkpoint_payload(epoch, global_step, best_f1, history, patience_counter):
    return {
        'epoch': epoch,
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_f1': best_f1,
        'history': history,
        'patience_counter': patience_counter,
        'rng_state': get_rng_state(),
        'run_dir': ACTIVE_RUN_DIR,
        'saved_at': datetime.now().isoformat(),
        'config': run_config,
    }

def save_last_checkpoint(epoch, global_step, best_f1, history, patience_counter):
    atomic_torch_save(
        checkpoint_payload(epoch, global_step, best_f1, history, patience_counter),
        LAST_CKPT_PATH,
    )

def save_best_checkpoint(epoch, global_step, best_f1, history, patience_counter):
    payload = checkpoint_payload(epoch, global_step, best_f1, history, patience_counter)
    atomic_torch_save(payload, BEST_CKPT_PATH)
    atomic_torch_save(payload['model_state_dict'], LEGACY_BEST_MODEL_PATH)

def evaluate_loader(loader, desc):
    if loader is None:
        return None

    model.eval()
    loss_sum = torch.tensor(0.0, device=device)
    predictions = []
    truths = []

    with torch.inference_mode():
        for inputs, labels in tqdm(loader, desc=desc, leave=False):
            inputs = inputs.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            labels = labels.to(device, non_blocking=True)
            with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            loss_sum += loss.detach() * labels.size(0)
            predictions.append(outputs.argmax(1).cpu())
            truths.append(labels.cpu())

    predictions = torch.cat(predictions)
    truths = torch.cat(truths)
    accuracy = (predictions == truths).float().mean().item()
    macro_f1 = f1_score(truths.numpy(), predictions.numpy(), average='macro')
    mean_loss = (loss_sum / len(loader.dataset)).item()
    return {
        'loss': mean_loss,
        'acc': accuracy,
        'f1': macro_f1,
    }

def train_one_epoch(global_step):
    model.train()
    loss_sum = torch.tensor(0.0, device=device)
    example_count = 0

    for inputs, labels in tqdm(train_loader, desc='TRAIN', leave=False):
        inputs = inputs.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            mixed_inputs, labels_a, labels_b, lam = mixup(inputs, labels)
            outputs = model(mixed_inputs)
            loss = mixup_loss(criterion, outputs, labels_a, labels_b, lam)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        loss_sum += loss.detach() * batch_size
        example_count += batch_size
        global_step += 1

    return (loss_sum / example_count).item(), global_step

model = build_model(freeze_backbone=FREEZE_BACKBONE)
optimizer = build_optimizer(model)
scheduler = build_scheduler(optimizer)
scaler = GradScaler(enabled=(device.type == 'cuda'))

start_epoch = 0
global_step = 0
best_f1 = 0.0
history = []
patience_counter = 0

if RESUME_FROM_LAST and os.path.exists(LAST_CKPT_PATH):
    try:
        checkpoint = torch.load(LAST_CKPT_PATH, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        restore_rng_state(checkpoint.get('rng_state'))
        start_epoch = checkpoint['epoch'] + 1
        global_step = checkpoint.get('global_step', start_epoch * max(1, len(train_loader)))
        best_f1 = checkpoint.get('best_f1', 0.0)
        history = checkpoint.get('history', [])
        patience_counter = checkpoint.get('patience_counter', 0)
        print(f'Resumed training from epoch {start_epoch + 1}.')
    except Exception as exc:
        print(f'Failed to resume from last checkpoint: {exc}')
        print('Starting a fresh run with the same run directory.')
else:
    print('Starting a fresh training run.')

def write_run_summary(extra=None):
    summary = {
        'active_run_dir': ACTIVE_RUN_DIR,
        'last_checkpoint_path': LAST_CKPT_PATH,
        'best_checkpoint_path': BEST_CKPT_PATH,
        'metrics_jsonl_path': METRICS_JSONL_PATH,
        'metrics_csv_path': METRICS_CSV_PATH,
        'target_total_epochs': TARGET_TOTAL_EPOCHS,
        'latest_completed_epoch': history[-1]['epoch'] if history else start_epoch,
        'best_f1': best_f1,
        'patience_counter': patience_counter,
        'updated_at': datetime.now().isoformat(),
    }
    if extra:
        summary.update(extra)
    atomic_write_json(RUN_SUMMARY_PATH, summary)

if start_epoch >= TARGET_TOTAL_EPOCHS:
    print(
        f'Checkpoint already reached epoch {start_epoch}. '
        'Increase TARGET_TOTAL_EPOCHS if you want to continue training further.'
    )
    write_run_summary({'status': 'already_complete'})
else:
    for epoch in range(start_epoch, TARGET_TOTAL_EPOCHS):
        train_started_at = time.time()
        train_loss, global_step = train_one_epoch(global_step)
        train_seconds = time.time() - train_started_at

        train_metrics = None
        val_metrics = None
        train_eval_seconds = 0.0
        val_seconds = 0.0
        should_validate = (epoch + 1) % VALIDATE_EVERY_N_EPOCHS == 0 or (epoch + 1) == TARGET_TOTAL_EPOCHS
        if should_validate:
            train_eval_started_at = time.time()
            train_metrics = evaluate_loader(train_eval_loader, 'TRAIN METRICS')
            train_eval_seconds = time.time() - train_eval_started_at
            val_started_at = time.time()
            val_metrics = evaluate_loader(val_loader, 'VAL')
            val_seconds = time.time() - val_started_at
            scheduler.step(val_metrics['f1'])

        improved = False
        if val_metrics is not None:
            if val_metrics['f1'] > best_f1:
                best_f1 = val_metrics['f1']
                patience_counter = 0
                improved = True
            else:
                patience_counter += 1

        current_lrs = [group['lr'] for group in optimizer.param_groups]
        epoch_record = {
            'epoch': epoch + 1,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'train_loss': round(float(train_loss), 6),
            'train_eval_loss': round(float(train_metrics['loss']), 6) if train_metrics else None,
            'train_acc': round(float(train_metrics['acc']), 6) if train_metrics else None,
            'train_f1': round(float(train_metrics['f1']), 6) if train_metrics else None,
            'val_loss': round(float(val_metrics['loss']), 6) if val_metrics else None,
            'val_acc': round(float(val_metrics['acc']), 6) if val_metrics else None,
            'val_f1': round(float(val_metrics['f1']), 6) if val_metrics else None,
            'best_f1': round(float(best_f1), 6),
            'patience_counter': int(patience_counter),
            'global_step': int(global_step),
            'lr_backbone': round(float(current_lrs[0]), 8),
            'lr_head': round(float(current_lrs[-1]), 8),
            'train_seconds': round(float(train_seconds), 2),
            'train_eval_seconds': round(float(train_eval_seconds), 2),
            'val_seconds': round(float(val_seconds), 2),
        }
        history.append(epoch_record)
        append_jsonl(METRICS_JSONL_PATH, epoch_record)
        write_csv(METRICS_CSV_PATH, history)

        save_last_checkpoint(epoch, global_step, best_f1, history, patience_counter)
        if improved:
            save_best_checkpoint(epoch, global_step, best_f1, history, patience_counter)

        write_run_summary({'status': 'training', 'latest_completed_epoch': epoch + 1})

        train_f1_display = f"{epoch_record['train_f1']:.4f}" if epoch_record['train_f1'] is not None else 'n/a'
        val_loss_display = f"{epoch_record['val_loss']:.4f}" if epoch_record['val_loss'] is not None else 'n/a'
        val_f1_display = f"{epoch_record['val_f1']:.4f}" if epoch_record['val_f1'] is not None else 'n/a'
        print(
            f"Epoch {epoch + 1}/{TARGET_TOTAL_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train F1: {train_f1_display} | "
            f"Val Loss: {val_loss_display} | "
            f"Val F1: {val_f1_display} | "
            f"Best F1: {best_f1:.4f} | "
            f"Train: {train_seconds:.1f}s | "
            f"Train Metrics: {train_eval_seconds:.1f}s | Val: {val_seconds:.1f}s"
        )

        if val_metrics is not None and patience_counter >= EARLY_STOP_PATIENCE:
            print(f'Early stopping triggered at epoch {epoch + 1}.')
            write_run_summary({'status': 'early_stopped', 'latest_completed_epoch': epoch + 1})
            break

evaluation_checkpoint_path = None
if os.path.exists(BEST_CKPT_PATH):
    evaluation_checkpoint_path = BEST_CKPT_PATH
elif os.path.exists(LAST_CKPT_PATH):
    evaluation_checkpoint_path = LAST_CKPT_PATH

if evaluation_checkpoint_path and test_loader is not None:
    evaluation_checkpoint = torch.load(evaluation_checkpoint_path, map_location=device)
    model.load_state_dict(evaluation_checkpoint['model_state_dict'])
    test_metrics = evaluate_loader(test_loader, 'TEST')
    write_run_summary({
        'status': 'finished',
        'test_loss': round(float(test_metrics['loss']), 6),
        'test_acc': round(float(test_metrics['acc']), 6),
        'test_f1': round(float(test_metrics['f1']), 6),
    })
    print(
        f"Final TEST | Loss: {test_metrics['loss']:.4f} | "
        f"Acc: {test_metrics['acc']:.4f} | F1: {test_metrics['f1']:.4f}"
    )
else:
    write_run_summary({'status': 'finished'})

## 6. Inference Function

In [ ]:
import os
import random

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.amp import autocast
from torchvision import models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'build_model' not in globals():
    def build_model(freeze_backbone=False):
        model = models.efficientnet_v2_s(weights='DEFAULT')
        for parameter in model.parameters():
            parameter.requires_grad = not freeze_backbone
        in_features = model.classifier[1].in_features
        model.classifier = nn.Linear(in_features, 2)
        return model.to(device)

checkpoint_candidates = []
if 'BEST_CKPT_PATH' in globals():
    checkpoint_candidates.append(BEST_CKPT_PATH)
if 'LAST_CKPT_PATH' in globals():
    checkpoint_candidates.append(LAST_CKPT_PATH)
checkpoint_candidates.extend([
    '/content/drive/MyDrive/ai_vs_real_model_checkpoints/best_model.pth',
    '/content/drive/MyDrive/ai_vs_real_model_checkpoints/best_checkpoint.pth',
])
MODEL_LOAD_PATH = next((path for path in checkpoint_candidates if os.path.exists(path)), None)

inference_model = build_model(freeze_backbone=False)
if MODEL_LOAD_PATH:
    loaded_object = torch.load(MODEL_LOAD_PATH, map_location=device)
    state_dict = loaded_object.get('model_state_dict', loaded_object) if isinstance(loaded_object, dict) else loaded_object
    inference_model.load_state_dict(state_dict)
    print(f'Loaded model from {MODEL_LOAD_PATH}')
else:
    print('Warning: No saved checkpoint was found. Inference is using an untrained model.')

inference_model.eval()
inference_transform = transforms.Compose([
    transforms.Resize(TRAIN_IMAGE_SIZE + 12),
    transforms.CenterCrop(TRAIN_IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict_image(image_path, mdl=inference_model, dev=device):
    image = Image.open(image_path).convert('RGB')
    tensor = inference_transform(image).unsqueeze(0).to(dev)
    with torch.no_grad(), autocast(device_type=dev.type, enabled=(dev.type == 'cuda')):
        logits = mdl(tensor)
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
    predicted_index = int(np.argmax(probabilities))
    predicted_label = 'AI-Generated' if predicted_index == 0 else 'Real'
    confidence = round(float(probabilities[predicted_index]) * 100, 2)
    return predicted_label, confidence

print('Inference demo (Randomly sampled from training data):')
for class_dir, expected_label in [(LOCAL_TRAIN_FAKE, 'AI-Generated'), (LOCAL_TRAIN_REAL, 'Real')]:
    if not os.path.exists(class_dir):
        print(f'Directory not found: {class_dir}')
        continue
    image_files = [name for name in os.listdir(class_dir) if name.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not image_files:
        print(f'No image files found in {class_dir}')
        continue
    for file_name in random.sample(image_files, min(2, len(image_files))):
        image_path = os.path.join(class_dir, file_name)
        predicted_label, confidence = predict_image(image_path)
        marker = '✅' if predicted_label == expected_label else '❌'
        print(f'  {marker} {predicted_label:14s} ({confidence:.1f}%)  actual={expected_label}  file={file_name}')

In [ ]:
from google.colab import files
import os

# 1. Upload the file
print('Please select an image file to check:')
uploaded = files.upload()

if uploaded:
    for fname in uploaded.keys():
        # 2. Run prediction using the model and function from Cell 06
        label, confidence = predict_image(fname)

        # 3. Display result
        print(f'\n--- RESULT ---')
        print(f'Prediction: {label}')
        print(f'Confidence: {confidence}%')
        print(f'File      : {fname}')

        # Clean up the uploaded file to keep /content tidy
        os.remove(fname)
else:
    print('No file uploaded.')